## 第9章 魔法方法、特性和迭代器


### 1.构造函数

- **定义**：在对象创建时自动调用的魔法方法，用于初始化对象状态。
    - 方法：`__init__(self, *args, **kwargs)`。
    - 继承：需在构造函数中调用父类构造函数，否则子类属性未初始化。
        - 新式方法：`super().__init__(args, kwargs)`。
        - 旧式方法：`ClassName.__init__(self, args, kwargs)`。

In [ ]:
# 父类
class Bird:
    # 构造函数
    def __init__(self):
        self.hungry = True
    def eat(self):
        if self.hungry:
            print('Aaaah ...')
            self.hungry = False
        else:
            print('No, thanks!')

# 子类
class SongBird(Bird):
    # 重写构造函数
    def __init__(self):
        super().__init__()      # 新式方法调用父类构造函数
        # Bird.__init__(self)   # 旧式方法调用父类构造函数
        self.sound = 'Squawk!'
    def sing(self):
        print(self.sound)

sb = SongBird()
sb.eat()
sb.sing()

### 2.元素访问

- 通过实现特定的魔法方法，可以让自定义对象的行为像序列或映射一样。主要包括如下魔术方法：
    - `__len__(self)`：返回对象的长度(如列表、元组、字符串等)。
    - `__getitem__(self, key)`：返回对象指定索引或键对应的值。
    - `__setitem__(self, key, value)`：设置对象指定索引或键对应的值。
    - `__delitem__(self, key)`：删除对象指定索引或键对应的值。

In [ ]:
# 通过实现__getitem__和__setitem__方法，创建一个具有序列特性的类
class ArithmeticSequence:
    def __init__(self, start=0, step=1):
        self.start = start
        self.step = step
        self.changed = {}  # 存储被修改过的值

    def _check_index(self, key):
        if not isinstance(key, int): raise IndexError
        if key < 0: raise IndexError

    # 实现__getitem__方法，返回指定索引对应的值
    def __getitem__(self, key):
        self._check_index(key)
        try:
            return self.changed[key]  # 修改过 → 返回修改值
        except KeyError:
            return self.start + key * self.step  # 未修改 → 计算值

    # 实现__setitem__方法，设置指定索引对应的值
    def __setitem__(self, key, value):
        self._check_index(key)
        self.changed[key] = value  # 修改值

s = ArithmeticSequence(1, 2)
print(s[4])     # 9
s[4] = 2
print(s[4])     # 2
print(s[5])     # 11


In [ ]:
# 继承内置类型比自己实现所有方法省力得多，且行为与原类型完全一致
class CounterList(list):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.counter = 0

    def __getitem__(self, key):
        self.counter += 1     # 每次访问计数+1
        return super(CounterList, self).__getitem__(key)

cl = CounterList(range(10))
print(cl[0] + cl[1])
print(cl.counter)  # 2

### 3.特性

- **定义**：通过存取方法定义的属性，**让你以属性语法访问方法**。
    - 方法一：`property(fget, fset, fdel, doc)`，fget为获取方法，fset为设置方法，fdel为删除方法，doc为文档字符串。
    - 方法二：使用__getattr__、__setattr__等魔法方法，自定义属性访问行为。
        - `__getattribute__(self, name)`：拦截所有属性访问(只适用新式类)。
        - `__getattr__(self, name)`：仅在属性访问**未找到**时调用。
        - `__setattr__(self, name, value)`：试图给属性赋值时调用。
        - `__delattr__(self, name)`：试图删除属性时调用。

In [ ]:
class Rectangle:
    def __init__(self, width=1, height=1):
        self.width = width
        self.height = height

    def set_size(self, size):
        self.width, self.height = size

    def get_size(self):
        return self.width, self.height

    # 定义特性
    size = property(get_size, set_size)

r = Rectangle(10, 5)
print(r.size)       # 像属性一样访问，实际调用 get_size 方法
r.size = (20, 10)   # 像属性一样赋值，实际调用 set_size 方法
print(r.width)

- **静态方法**：无 `self` 参数，与类无关的实用函数，通过`@staticmethod`装饰器定义。
- **类方法**：第一个参数是类本身(`cls`)，而非实例，通过`@classmethod`装饰器定义。

In [ ]:
class MyClass:
    # 静态方法
    @staticmethod
    def smeth():
        print('This is a static method')

    # 类方法
    @classmethod
    def cmeth(cls):
        print('This is a class method of', cls)

MyClass.smeth()
MyClass.cmeth()

### 4.迭代器

- **定义**：迭代器是一种特殊类型的对象，实现了`__iter__`方法的对象是可迭代的，而实现了`__next__`方法的对象是迭代器，可以使用`iter()`函数将可迭代对象转换为迭代器。
    - `__iter__(self)`：返回迭代器对象，其中包含`__next__`方法。
    - `__next__(self)`：返回下一个元素，直到没有更多元素时抛出`StopIteration`。也可以使用`next()`函数调用。

In [ ]:
class Fibs:
    def __init__(self):
        self.a, self.b = 0, 1
    def __iter__(self):  # 可迭代
        return self
    def __next__(self):  # 迭代器
        self.a, self.b = self.b, self.a + self.b
        return self.a

# 找出第一个大于1000的斐波那契数
fibs = Fibs()
for f in fibs:
    if f > 1000:
        print(f)
        break

In [ ]:
it = iter([1, 2, 3, 4, 5])  # 将内置类型转为迭代器
print(next(it))  # 获取下一个元素
print(next(it))  # 获取下一个元素

In [ ]:
# for循环底层机制
# for x in range(10) 类似以下代码
it = iter(range(10))        # 调用 __iter__
while True:
    try:
        x = next(it)        # 调用 __next__
    except StopIteration:
        break

### 5.生成器

- **定义**：生成器是一种使用普通函数语法(包含`yield`语句)定义的迭代器。
- **机制**：调用生成器函数不会执行函数体，而是返回一个生成器迭代器。每次执行到`yield`时，生成一个值并暂停执行；被`next()`函数唤醒后从暂停处继续执行。
- **工作流程**：<br>
```text
调用生成器函数 → 返回生成器迭代器(不执行函数体)
   ↓
next() → 执行到第一个 yield → 返回值 → 暂停
   ↓
next() → 从暂停处继续 → 执行到下一个 yield → 返回值 → 暂停
   ↓
......
   ↓
函数结束或 return → 引发 StopIteration
```

In [ ]:
# 递归式生成器，用于扁平化任意嵌套层列表
def flatten(nested):
    try:
        # 不迭代类似于字符串的对象
        try:
            nested + '' # type: ignore 测试是否像字符串一样可以拼接
        except TypeError:
            pass
        else:
            raise TypeError     # 像字符串 → 抛出异常，不迭代

        for sublist in nested:
            for element in flatten(sublist):     # 递归调用，处理子列表
                yield element
    except TypeError:
        yield nested        # 不可迭代 → 作为值生成

print(list(flatten(['foo', ['bar', ['baz']]])))             # ['foo', 'bar', 'baz']
print(list(flatten([[[1], 2], 3, 4, [5, [6, 7]], 8])))      # [1, 2, 3, 4, 5, 6, 7, 8]

- **yield vs return**：
    - `yield`：暂停函数，生成一个值，下次调用从暂停处继续执行。
    - `return`：结束函数执行，包含整个迭代过程，返回一个值。

In [ ]:
def list2list(nested):
    for sublist in nested:
        for element in sublist:
            yield element
            if element > 9:  # 当元素大于9时，结束迭代
                return

list(list2list([[1, 2], [3, 4], [8, 10], [3, 2], [5]]))  # [1, 2, 3, 4, 8, 10]，当元素大于9时，因return语句结束迭代，后续元素不被处理

- **生成器表达式**：用圆括号`()`包裹、惰性求值(列表推导式用方括号`[]`包裹、立即求值)。

In [ ]:
sum(i**2 for i in range(10))     # 直接在函数调用中使用，无需额外括号

- **通用生成器**：由两部分构成，生成器函数(包含`yield`的`def`语句)和生成器迭代器(调用生成器函数返回的对象)。


In [ ]:
def generator():
    for i in range(10):
        yield i

print(generator)        # <function generator at 0x...>，生成器函数
print(generator())      # <generator object generator at 0x...>，生成器迭代器

- **生成器方法**：在生成器开始运行后，可使用生成器和外部之间的通信渠道向它提供值，或从它获取值。
    - `send()`：将值传给生成器内部挂起的`yield`表达式(此时`yield`作为表达式返回传入的值)。
        - 注意：刚启动的生成器只能传`None`或使用`next()`。
    - `throw()`：在`yield`处引发指定异常。
    - `close()`：停止生成器，在`yield`处引发`GeneratorExit`异常。若需清理资源，可将`yield`置于`try/finally`语句中。


In [ ]:
def repeater(value):
    while True:
        new = (yield value)                 # yield 作为表达式，接收 send 的值
        if new is not None: value = new

r = repeater(42)
print(next(r))          # 42
print(r.send(100))      # 100
print(r.send(200))      # 200

### 6.本章小结

```text
魔法方法、特性和迭代器
│
├── 构造函数 __init__
│   ├── 对象创建后自动调用
│   ├── 重写时必须调用超类构造函数
│   └── super().__init__() ⭐
│
├── 元素访问（序列/映射协议）
│   ├── __len__ / __getitem__ / __setitem__ / __delitem__
│   ├── 继承内置类型（list/dict/str）→ 捷径
│   └── 负索引、TypeError、IndexError 规范
│
├── 特性 property ⭐
│   ├── property(fget, fset, fdel, doc)
│   ├── 像属性一样访问，底层调用方法
│   ├── @staticmethod / @classmethod
│   └── __getattr__ / __setattr__ / __delattr__
│
├── 迭代器 ⭐
│   ├── __iter__() → 返回迭代器
│   ├── __next__() → 返回下一个值 / StopIteration
│   ├── iter() / next() 内置函数
│   └── for 循环的底层机制
│
└── 生成器 ⭐
    ├── yield 暂停/恢复
    ├── 生成器推导 (expr for ...)
    ├── 递归式生成器
    ├── send() / throw() / close()
    └── 惰性计算，省内存
```